In [1]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

# path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
# grewpy.set_config('ud')
path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_Romanian-RRT"
grewpy.set_config('ud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

connected to port: 56472


In [2]:
all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])

In [3]:
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value

In [4]:
print(len(matches))

2162


In [5]:
# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)

In [6]:
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

for key, value in match_upos.items():
    print(key, len(value))

('țăran', 'NOUN') 15
('ține', 'VERB') 67
('țigară', 'NOUN') 11
('țesut', 'NOUN') 16
('țară', 'NOUN') 188
('știință', 'NOUN') 19
('științific', 'ADJ') 40
('ști', 'VERB') 142
('șterge', 'VERB') 11
('șold', 'NOUN') 12
('șir', 'NOUN') 30
('și', 'CCONJ') 5312
('șef', 'NOUN') 22
('ședință', 'NOUN') 11
('școală', 'NOUN') 38
('șase', 'NUM') 38
('șapte', 'NUM') 30
('Ștefan', 'PROPN') 17
('înțelegere', 'NOUN') 16
('înțelege', 'VERB') 50
('înălțime', 'NOUN') 11
('învățământ', 'NOUN') 18
('învăța', 'VERB') 15
('întări', 'VERB') 26
('întârzia', 'VERB') 12
('întâmplare', 'NOUN') 18
('întâmpla', 'VERB') 54
('întâlnire', 'NOUN') 19
('întâlni', 'VERB') 36
('întâi', 'ADV') 13
('întâi', 'NUM') 15
('întuneric', 'NOUN') 12
('întrucât', 'SCONJ') 11
('întru', 'ADP') 332
('întreținere', 'NOUN') 14
('întreține', 'VERB') 14
('întrerupere', 'NOUN') 30
('întrerupe', 'VERB') 31
('întreprindere', 'NOUN') 21
('întreprinde', 'VERB') 17
('întregime', 'NOUN') 14
('întreg', 'ADJ') 83
('întrebare', 'NOUN') 17
('întreba',

In [7]:
with open("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

In [8]:
data = { k : list() for k in match_upos }
for node, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[node].append(formatted_features)

In [9]:
unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [10]:
unique_features

['node:X:child:Abbr=Yes',
 'node:X:child:AdpType=Prep',
 'node:X:child:Case=Acc',
 'node:X:child:Case=Acc,Nom',
 'node:X:child:Case=Dat',
 'node:X:child:Case=Dat,Gen',
 'node:X:child:Case=Gen',
 'node:X:child:Case=Nom',
 'node:X:child:Case=Voc',
 'node:X:child:Definite=Def',
 'node:X:child:Definite=Ind',
 'node:X:child:Degree=Cmp',
 'node:X:child:Degree=Pos',
 'node:X:child:Degree=Sup',
 'node:X:child:Gender=Fem',
 'node:X:child:Gender=Masc',
 'node:X:child:Mood=Imp',
 'node:X:child:Mood=Ind',
 'node:X:child:Mood=Sub',
 'node:X:child:NumForm=Combi',
 'node:X:child:NumForm=Digit',
 'node:X:child:NumForm=Roman',
 'node:X:child:NumForm=Word',
 'node:X:child:NumType=Card',
 'node:X:child:NumType=Ord',
 'node:X:child:Number=Plur',
 'node:X:child:Number=Sing',
 'node:X:child:Number__psor=Plur',
 'node:X:child:Number__psor=Sing',
 'node:X:child:PartType=Inf',
 'node:X:child:Person=1',
 'node:X:child:Person=2',
 'node:X:child:Person=3',
 'node:X:child:Polarity=Neg',
 'node:X:child:Polarity=Pos

In [11]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(2198, 573)


In [12]:
import numpy as np
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

def find_optimal_clusters(X, max_clusters=20, metric='cosine', method='complete'):
    distance_matrix = pdist(X, metric=metric)
    linked = linkage(distance_matrix, method=method, optimal_ordering=True)
    
    silhouette_scores = []
    for num_clusters in range(2, max_clusters + 1):
        labels = fcluster(linked, num_clusters, criterion='maxclust')
        if len(np.unique(labels)) > 1:  # Ensure there is more than one cluster
            score = silhouette_score(X, labels, metric=metric)
            silhouette_scores.append(score)
            print(f'Number of clusters: {num_clusters}, Silhouette Score: {score}')
        else:
            silhouette_scores.append(-1)  # Append a low score if only one cluster
    
    optimal_clusters = np.argmax(silhouette_scores) + 2  # +2 because range starts from 2
    return optimal_clusters, silhouette_scores

# Find the optimal number of clusters
optimal_clusters, silhouette_scores = find_optimal_clusters(X, max_clusters=50)
print(f'Optimal number of clusters: {optimal_clusters}')
print(X.shape)

Number of clusters: 2, Silhouette Score: 0.2796810241060291
Number of clusters: 3, Silhouette Score: 0.30719890016670226
Number of clusters: 4, Silhouette Score: 0.3154279776670375
Number of clusters: 5, Silhouette Score: 0.42402680038980245
Number of clusters: 6, Silhouette Score: 0.4722792393256292
Number of clusters: 7, Silhouette Score: 0.48113696478181683
Number of clusters: 8, Silhouette Score: 0.46924246655379903
Number of clusters: 9, Silhouette Score: 0.49736928467851615
Number of clusters: 10, Silhouette Score: 0.5126048979505907
Number of clusters: 11, Silhouette Score: 0.5194588141072799
Number of clusters: 12, Silhouette Score: 0.4718442250878482
Number of clusters: 13, Silhouette Score: 0.4671506128530967
Number of clusters: 14, Silhouette Score: 0.46942743150872845
Number of clusters: 15, Silhouette Score: 0.4704859205213365
Number of clusters: 16, Silhouette Score: 0.46890931563261445
Number of clusters: 17, Silhouette Score: 0.3248128280817357
Number of clusters: 18, S

In [13]:
distance_matrix = pdist(X, metric='cosine')
linked = linkage(distance_matrix, method="complete", optimal_ordering=True)
labels = fcluster(linked, optimal_clusters, criterion='maxclust')
clusters = {i: [] for i in range(1, optimal_clusters + 1)}
for i, label in enumerate(labels):
    clusters[label].append(i)

In [14]:
for cluster, members in clusters.items():
    print(f'Cluster {cluster}:')
    for member in members:
        print(f'  {unique_lemma[member]}')

Cluster 1:
  ('a', 'PART')
  ('avea', 'AUX')
  ('ca', 'SCONJ')
  ('ci', 'CCONJ')
  ('că', 'SCONJ')
  ('căci', 'CCONJ')
  ('dacă', 'SCONJ')
  ('dar', 'CCONJ')
  ('deci', 'CCONJ')
  ('deoarece', 'SCONJ')
  ('deși', 'SCONJ')
  ('el', 'PRON')
  ('eu', 'PRON')
  ('fi', 'AUX')
  ('fie', 'CCONJ')
  ('fiindcă', 'SCONJ')
  ('fără', 'SCONJ')
  ('nu', 'PART')
  ('ori', 'CCONJ')
  ('până', 'SCONJ')
  ('sau', 'CCONJ')
  ('sine', 'PRON')
  ('să', 'PART')
  ('tu', 'PRON')
  ('vrea', 'AUX')
  ('încât', 'SCONJ')
  ('însă', 'CCONJ')
  ('întrucât', 'SCONJ')
  ('și', 'CCONJ')
Cluster 2:
  ('acela', 'PRON')
  ('acesta', 'PRON')
  ('altul', 'PRON')
  ('care', 'PRON')
  ('ce', 'PRON')
  ('celălalt', 'PRON')
  ('ceva', 'PRON')
  ('cine', 'PRON')
  ('cineva', 'PRON')
  ('câtva', 'PRON')
  ('dumneata', 'PRON')
  ('dumneavoastră', 'PRON')
  ('fiecare', 'PRON')
  ('lui', 'PRON')
  ('meu', 'PRON')
  ('mult', 'PRON')
  ('nimeni', 'PRON')
  ('nimic', 'PRON')
  ('oricare', 'PRON')
  ('orice', 'PRON')
  ('său', 'PRON'

In [15]:
pie_chart = {}
for cluster, members in clusters.items():
    pie_chart[cluster] = {}
    for member in members:
        if unique_lemma[member][1] in pie_chart[cluster]:
            pie_chart[cluster][unique_lemma[member][1]] += 1
        else:
            pie_chart[cluster][unique_lemma[member][1]] = 1

In [20]:
# make pie chart with plotly
import plotly.express as px
import plotly.graph_objects as go

# Function to create pie chart for a given cluster
def create_pie_chart(cluster_number):
    labels = [f'{k} ({v})' for k, v in pie_chart[cluster_number].items()]
    values = list(pie_chart[cluster_number].values())
    fig = go.Figure(data=[go.Pie(labels=labels, values=values)])
    return fig

# Create initial pie chart
fig = create_pie_chart(1)

# Add dropdown menu
dropdown_buttons = [
    {
        'label': f'Cluster {i}',
        'method': 'update',
        'args': [{'values': [list(pie_chart[i].values())], 'labels': [[f'{k} ({v})' for k, v in pie_chart[i].items()]]}]
    } for i in pie_chart.keys()
]

fig.update_layout(
    updatemenus=[
        {
            'buttons': dropdown_buttons,
            'direction': 'down',
            'showactive': True,
        }
    ]
)

fig.show()

In [21]:
fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/output/visualisations/Romanian_all_lex_units_pie.html")

In [17]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

In [22]:


# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': labels,
    'Word': unique_lemma
})
# pca1_min, pca1_max = df['PCA1'].min(), df['PCA1'].max()
# pca2_min, pca2_max = df['PCA2'].min(), df['PCA2'].max()
# Create a scatter plot with Plotly
fig = go.Figure()

# Add traces for each cluster
for cluster in range(1, optimal_clusters + 1):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

# Update layout with dropdown menu
fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    width=1600,  # Set the width of the figure
    height=800,  # Set the height of the figure
    # xaxis=dict(range=[pca1_min, pca1_max]),
    # yaxis=dict(range=[pca2_min, pca2_max]),
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * optimal_clusters},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(optimal_clusters)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, optimal_clusters + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

In [23]:
fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/output/visualisations/Romanian_all_lex_units_tsne.html")

In [19]:
/Users/madalina/Documents/M1TAL/stage-SK/wolof_all_categs.ipynb

NameError: name 'Users' is not defined